> [!IMPORTANT]
> **Disclaimer**: The content and views presented during this session are the author's own and not of any organizations they are associated with or employed at. The code shown in this repository is for illustration and educational purposes only. It is not production-grade; error handling, security, and scalability are not fully addressed.

# Lab 1: Structured Logging, FinOps, and GreenOps for AI Agents
**Faculty Development Programme on Observability for AI Agents**

> [!NOTE]
> **Curriculum Cross-Reference**: This laboratory exercise implements the practical aspects of the **Structured Logging, FinOps, and GreenOps** slides detailed in **Section 2 of [00_curriculum_and_agenda.ipynb](00_curriculum_and_agenda.ipynb)**.

### Overview
This laboratory exercise covers the fundamentals of structured logging using Python's `structlog` library, extending the telemetry pipeline into both **FinOps (Cost Tracking)** and **GreenOps (Carbon Emissions Tracking)**.

### 🤖 The Telemetry Paradigm Shift: LLMs vs. AI Agents
Traditional application monitoring focuses on infrastructure health (CPU, memory, and database IO). The emergence of large language models shifted telemetry toward tracking stateless inputs, output tokens, and prompt generation costs.
However, **autonomous AI agents** introduce a higher dimension of runtime complexity:
* **LLM Applications**: Characterised by stateless, single-turn (where a single prompt receives a single completion without memory of past messages) completions. The request-response cycle is linear, enabling straightforward metric mapping (one input, one output).
* **AI Agents**: Characterised by stateful execution loops (where the system retains memory of past interactions to make subsequent decisions, e.g., ReAct (Reasoning and Acting) or Plan-and-Solve architectures) that recursively call LLMs to plan tasks, invoke external tools (invoking external functions or databases), process responses, and repeat the loop until achieving a termination condition.
* **Agent Observability Requirements**: An agent execution run is not a single transaction. It is a hierarchical tree of nested operations (thought cycles, tool invocations, and sequential model calls). Trace logging must correlate all nested steps to a single session context, tracking cumulative token expenditures (FinOps) and the net carbon footprint (GreenOps) of the reasoning loop.

### 🍃 The Case for GreenOps in AI systems
Generative AI workloads are exceptionally compute-intensive. While environmental impact analysis often emphasizes model training, tracking model inference energy consumption is equally critical. Inference workloads represent the vast majority of an AI model's lifetime energy footprint. This laboratory exercise utilizes **CodeCarbon**—an open-source, local-first library—to measure real-time CPU/GPU energy consumption and estimate carbon emissions ($CO_2e$) for agent runs, injecting these parameters into structured JSON logs.

### Learning Objectives:
1. Distinguish between traditional APM, stateless LLM monitoring, and stateful agent loop observability.
2. Configure `structlog` to output standardized, machine-readable JSON log events.
3. Instrument client wrappers to record execution latency, token usage, and cumulative costs.
4. Integrate `codecarbon` in offline mode to measure local hardware energy consumption and carbon emissions.
5. Bind session context details alongside cost and environmental metrics within structured log outputs.

## 1. Setup and Environment
Required Python modules must be installed first. If running locally, the block below installs `structlog` and other dependencies.

In [ ]:
# Install required packages
%pip install -r ../requirements.txt

## 2. Why Structured Logging?

Traditional Python logging outputs text strings:
`INFO:root:User user123 requested 'explain quantum computing' at 12:34 PM. Cost was 0.003`

While readable by humans, it is difficult for automated log management systems to answer questions such as:
* *What is the average cost for user123?*
* *What was the total latency across all requests today?*

Querying this requires complex regular expressions.

**Structured Logging** converts output to:
```json
{
  "event": "llm_completion",
  "level": "info",
  "timestamp": "2026-05-22T13:40:00Z",
  "user_id": "user123",
  "prompt": "explain quantum computing",
  "latency_seconds": 1.42,
  "cost": 0.0034,
  "tokens_input": 5,
  "tokens_output": 48
}
```
Now, any log analytics platform can index and query these fields directly.

## 3. Configuring `structlog` for JSON Output

The `structlog` library is configured to output clean JSON logs. Standard processors are set up to inject timestamps and format the output.

In [ ]:
import sys
import os
import time
import logging
from typing import Optional
# Add parent directory to sys.path so we can import src modules
sys.path.append(os.path.abspath('..'))

# Ingests Python's structured logging library for machine-readable JSON outputs
import structlog
# Imports local mock client to simulate model inferences, token metrics, and cost tracking offline
from src.mock_llm import MockLLMClient, MockLLMResponse
# Imports CodeCarbon tracker to measure CPU/GPU energy consumption and estimate carbon footprint
from codecarbon import OfflineEmissionsTracker

# Configure structlog
# Production Tip: In high-throughput deployments, JSON logs generate massive volumes.
# Integrate log-level filtering at the application boundary (e.g., using structlog's
# filtering processors or in the forwarder daemon) to prevent logging debug events in production.
structlog.configure(
    processors=[
        # Injects the severity level (e.g., info, warning, error) into the log payload dictionary
        structlog.processors.add_log_level,
        # Appends an ISO-8601 standardized timestamp key to ensure accurate temporal sequencing
        structlog.processors.TimeStamper(fmt="iso"),
        # Format the log output as a JSON dictionary
        structlog.processors.JSONRenderer()
    ],
    # Specifies the underlying datatype (a standard Python dictionary) used to hold log context keys
    context_class=dict,
    # Directs structlog to print the resulting JSON string to stdout (console output)
    logger_factory=structlog.PrintLoggerFactory(),
    # Caches the constructed logger instance on first lookup to avoid performance overhead in subsequent calls
    cache_logger_on_first_use=True,
)

logger = structlog.get_logger()

# Test log
logger.info("logging_initialized", status="success", version="1.0.0")

### 🤖 Local Prerequisite: The Mock LLM Client
To ensure this workshop is completely **offline, local-first, and zero-cost**, the code does not query commercial external API providers (e.g., OpenAI or Gemini) which might require dynamic sign-up credentials and credit card billing authorizations. 

Instead, a simulated model client module (**`MockLLMClient`** located in [src/mock_llm.py](../src/mock_llm.py)) is used:
* **Simulated Inference**: It resolves prompts locally and yields natural language completions based on predefined search and math templates.
* **Latency Emulation**: It injects realistic processing latency using a baseline delay combined with random variance (jitter) to simulate server-side network lag.
* **Token and Cost Tracking**: It calculates prompt and completion token volumes using typical text ratios and maps them to financial costs.
* **Observability Integration**: It returns a structured `MockLLMResponse` object containing the text payload alongside numeric metadata (`prompt_tokens`, `completion_tokens`, and `cost`) to populate structured JSON logs.

## 4. Instrumenting LLM Cost, Latency, and Carbon Emissions Tracking

An enhanced function `generate_with_logging` is constructed to wrap the `MockLLMClient`.
The function:
1. Initializes a timer.
2. Configures **CodeCarbon's `OfflineEmissionsTracker`** in offline mode (using country code 'IND' to bypass external GeoIP lookup operations).
3. Executes the LLM call.
4. Measures the carbon emissions dynamically.
5. Computes execution duration (latency).
6. Extracts token usage and estimated cost parameters (FinOps).
7. Emits a single structured JSON log containing all these dimensions (FinOps and GreenOps combined).

In [ ]:
# The MockLLMClient is initialized to simulate inference completions
llm_client: MockLLMClient = MockLLMClient()

# CodeCarbon verbose logging is suppressed to prevent cluttering stdout
logging.getLogger("codecarbon").setLevel(logging.ERROR)

def generate_with_logging(prompt: str, user_id: str, session_id: str) -> str:
    """
    Executes an LLM inference request, measures latency and energy draw,
    and records findings using structured JSON log emissions.

    Args:
        prompt (str): The input prompt string.
        user_id (str): Unique identifier of the user initiating the request.
        session_id (str): Session context identifier for tracking multi-turn loops.

    Returns:
        str: The raw text response generated by the model.

    Raises:
        Exception: Re-raises any exceptions encountered during the generation step.
    """
    # Request-level log context is established by binding parameters
    log = logger.bind(user_id=user_id, session_id=session_id)

    # Production Tip: Sparse Event Schemas. Attributes like cost, latency, and carbon emissions
    # are omitted from the start event. Emitting keys only when they contain valid values reduces
    # log payload sizes, decreases bandwidth, and lowers indexer storage costs.
    log.info("llm_generation_started", prompt_preview=prompt[:60])

    # OfflineEmissionsTracker is configured for local carbon tracking
    tracker = OfflineEmissionsTracker(
        country_iso_code="IND",
        log_level="error"
    )

    start_time: float = time.time()
    tracker.start()
    try:
        response: MockLLMResponse = llm_client.generate(prompt)

        latency: float = time.time() - start_time
        emissions_kg: Optional[float] = tracker.stop()

        # Ingestion metrics are recorded into JSON logs
        log.info(
            "llm_generation_completed",
            latency_seconds=round(latency, 3),
            tokens_input=response.prompt_tokens,
            tokens_output=response.completion_tokens,
            cost=round(response.cost, 6),
            co2_emissions_kg=round(emissions_kg if emissions_kg else 0.0, 10),
            response_preview=response.text[:60]
        )
        return response.text

    except Exception as e:
        latency = time.time() - start_time
        tracker.stop()
        log.error(
            "llm_generation_failed",
            latency_seconds=round(latency, 3),
            error=str(e)
        )
        raise

## 5. Simulating User Sessions

Let's simulate multiple requests from different users to see how the logs are produced. Notice how easy it is to trace log messages related to a single `session_id` or `user_id` in a JSON log stream.

In [ ]:
# Simulate user 1 asking a question
print("--- User 1, Request 1 ---")
response_1 = generate_with_logging(
    prompt="Explain quantum computing in simple terms.",
    user_id="user_001",
    session_id="session_abc_123"
)
print(f"User 1 Response: '{response_1}'")

# Simulate user 2 asking a question
print("\n--- User 2, Request 1 ---")
response_2 = generate_with_logging(
    prompt="What is the capital of France?",
    user_id="user_002",
    session_id="session_xyz_789"
)
print(f"User 2 Response: '{response_2}'")

> [!NOTE]
> **Observability Insight: Understanding Carbon Emission Measurements ($CO_2e$)**
> * **The Observation**: It is observed that User 2's carbon emissions (e.g., $7.82 \times 10^{-6}$ kg) are higher than User 1's emissions (e.g., $6.81 \times 10^{-6}$ kg), even though User 1 processed significantly more tokens (45 output tokens vs. 7 output tokens).
> * **The Cause (Duration vs. Compute)**: CodeCarbon calculates emissions based on active hardware power draw (in Watts) multiplied by the execution duration (latency). In this local laboratory setup:
>   - The model is a **mock client** that simulates latency using sleep delays (`time.sleep`) rather than running intensive GPU matrix multiplications. 
>   - Because the hardware remains in an idle power state for both requests, the calculated emissions are driven entirely by the **duration of the sleep timer**. Since User 2's response latency happened to be longer (1.49s vs 1.30s), it accumulated more energy usage.
> * **Production Reality**: In live environments with real LLMs (e.g., local Ollama running on a GPU or server CPU), the active power draw spikes dramatically during token generation (e.g., drawing 250W during generation vs 20W idle). In such setups, carbon emissions correlate strongly with token volume and active tensor processing compute cycles, rather than simple elapsed idle time.

## 6. Production Guidelines and Leading Practices

When deploying structured logging, FinOps, and GreenOps tracking in enterprise environments, the following practices are recommended:

### 📋 Structured Logging in Production
* **Asynchronous Log Writing**: Writing logs directly to stdout or disk synchronously introduces I/O blocking. Production applications should write logs asynchronously to stdout and run a dedicated daemon agent (e.g., **Vector** or **FluentBit**) to ship logs to central search repositories (e.g., Elasticsearch, Grafana Loki, or Google Cloud Logging).
* **Sensitive Data Protection and PII Redaction**: Real-world prompt contexts often contain sensitive user details (Personally Identifiable Information - PII) or credentials. To ensure Data Loss Prevention (DLP), production architectures integrate:
  * **[Google Cloud Sensitive Data Protection](https://cloud.google.com/sensitive-data-protection)**: A fully managed enterprise service to discover, classify, and redact sensitive data elements (such as credit cards, government IDs, and credentials) dynamically in AI inputs and completions.
  * **Open-Source Detection Libraries**: General open-source frameworks like **[Microsoft Presidio](https://github.com/microsoft/presidio)** (utilizing regular expressions, checksums, and Named Entity Recognition models to mask PII), **[Scrubadub](https://github.com/scrubadub/scrubadub)** (a lightweight Python PII scrubbing library), and input/output guardrail suites like **[Nvidia NeMo Guardrails](https://github.com/NVIDIA/NeMo-Guardrails)**.
  * *Implementation*: Configure these DLP processors as custom middleware or logging pipeline stages (e.g., a `structlog` processor) to automatically sanitize data before logs are serialized to stdout or files.
* **Dynamic Log Levels and Volume Management**: Structured JSON logs (especially those containing trace metadata and emissions metrics) generate massive storage volumes. Production systems establish strict log level definitions, run log level filters at the shipper daemon layer, and expose dynamic HTTP endpoints to alter log levels on demand.
* **Sparse Event Schemas (Key Field Relevance)**: Avoid writing empty keys or `null` values for attributes that are irrelevant to specific events (e.g., omitting `cost` or `co2_emissions_kg` keys on `llm_generation_started` logs). Logging clients should emit sparse JSON payloads where keys exist only if they contain valid metrics, reducing log storage size and indexing overhead.

### 💰 Production FinOps (Cost Controls)
* **API Rate Limiting and Token Quotas**: Enforce strict token budget limits per user or session to prevent loops from exhausting commercial LLM billing balances.
* **Prometheus Alerting Integration**: Export cost metrics as Prometheus counters and configure alerts (via Prometheus Alertmanager) to trigger pager notifications when total session expenditure exceeds configured limits.
* **Semantic Caching and Cost Savings Telemetry**: Implement semantic prompt caching (e.g., using **[GPTCache](https://github.com/zilliztech/GPTCache)** or vector databases) to intercept equivalent customer queries. Telemetry configurations must calculate and log the exact costs and tokens *saved* per cache hit, allowing the SRE dashboard to visualize the net financial savings of caching.

### 🍃 Production GreenOps (Environmental Footprint)
* **Carbon-Aware Scheduling (Marginal Intensity Remediations)**:
  * *How to Track*: Production orchestrators query grid APIs like **[Electricity Maps](https://www.electricitymaps.com/)** or **[WattTime](https://www.watttime.org/)** to fetch real-time and forecasted marginal carbon intensity ($gCO_2e/kWh$). Marginal intensity measures the emissions of the specific power plant (often fossil-fueled peakers) that responds to a change in electrical demand.
  * *What Happens During High-Intensity*: Running heavy AI workloads during peak intensity periods forces the grid to activate backup carbon-heavy generators (such as coal or natural gas peaker plants). This inflates corporate carbon footprints, degrades sustainability compliance ratings under standards like the GHG Protocol, and increases stress on congested local grids.
  * *Remediation*: Configure automated job queues to pause or delay non-critical, compute-heavy tasks (such as batch agent offline evaluations or vector indexing) until grid intensity drops below a defined threshold (e.g., during peak solar or wind generation hours).
* **Region Selection**: Deploy serverless agent workloads in green regions (cloud data centers powered primarily by carbon-free energy sources).
* **Carbon Savings Telemetry**: Track and report carbon emission savings ($CO_2e$ offset) achieved by cache hits that avoided GPU/CPU inference processing cycles.

### Summary of Lab 1
1. **Machine-parseable logs**: By using JSON output, ingestion systems can easily index and query fields without regular expressions.
2. **Cost and Performance tracking (FinOps)**: Ingesting structured logs allows monitoring of cumulative expenditure, latencies, and token volume per user or model version.
3. **Carbon Footprint tracking (GreenOps)**: Using CodeCarbon enables tracking the ecological impact of AI agent runs dynamically, allowing the development of energy-aware agent flows.
4. **Trace correlation**: Attaching `session_id` allows reconstructive trace stitching across distributed log collectors.

The subsequent lab covers **Distributed Tracing** to visualize multi-step agent decisions dynamically using OpenTelemetry and Arize Phoenix.